In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from analysis_scripts.graphs import set_size
from analysis_scripts.load_lammps import load_lammps
from analysis_scripts.dump import dump

plt.style.use('seaborn')
plt.style.use('dark_background')

# plt.style.use('tex')
dumpsdir = os.path.abspath("../dumps/")

In [ ]:
nsmc = 150
testname = "test_poly.m50.n1000.rho0.05.lp3.linear_prob1_rate10000_cutoff1.5_nsmc" + str(nsmc)
testype = "parallel"


bonds_d = dump(os.path.join(dumpsdir, f"{testname}_bond_{testype}.lammpstrj"))
coord_d = dump(os.path.join(dumpsdir, f"{testname}_dump_{testype}.lammpstrj"))
coord_d.sort()
try:
    coord_d.unwrap()
except Exception:
    print("Already unwrapped")


In [ ]:
bonds = bonds_d.all_atoms_info()[:, :, 1:]
bonds = np.concatenate(
    (np.repeat(np.full(bonds.shape[0], 1).cumsum(), bonds.shape[1]).reshape(
        bonds.shape[0], bonds.shape[1], 1
    ),
    bonds),
    axis=2,
)
itype = coord_d.all_atoms_info()[:, :, :6]
itype = np.concatenate(
    (np.repeat(np.full(itype.shape[0], 1).cumsum(), itype.shape[1]).reshape(
        itype.shape[0], itype.shape[1], 1
    ),
    itype),
    axis=2,
)

In [ ]:
itype2 = itype[(itype[:,:,3]==2)]
bonds2 = bonds[(bonds[:,:,3]==2) + (bonds[:,:,3]==3)]

typedf = pd.DataFrame(itype2, columns=["time","id","mol","type","x","y","z"])
bonds2df = pd.DataFrame(bonds2, columns=["time","bead1", "bead2","type"])
bonds3df = pd.DataFrame(bonds2, columns=["time","bead1", "bead2","type"])

plt.figure(figsize=set_size(500))
plt.plot(bonds2df[["time","type"]].groupby("time").count())
plt.plot(typedf[["time","type"]].groupby("time").count())

bonds2 = np.sort(bonds2.reshape(-1,nsmc,4),axis=1)
itype2 = itype2.reshape(-1,nsmc,2,7)

In [ ]:
fig, axes = plt.subplots(2,5,figsize=set_size(1500, subplots=(2,5)), sharex=True)

for j,ax in enumerate(axes.flatten()):
    ax.set_xlabel("Time [It]")
    ax.set_ylabel("Indexes")

    for i in range(nsmc):
        ax.fill_between(bonds2[:,i,0],bonds2[:,i,2],bonds2[:,i,1], alpha =0.5, color="grey")
        
        ax.plot(itype2[:,i,0,0],itype2[:,i,0,1], color="yellow", alpha=0.6, linestyle='-', marker='o', markersize=2)
        ax.plot(itype2[:,i,0,0],itype2[:,i,1,1], color="cyan", alpha=0.6,linestyle='-', marker='o', markersize=2)
    ax.set_ylim([j*5000,(j+1)*5000])

plt.savefig(f"results/kymograph_{testname}_{testype}.pdf")